# Stage 2: sample thermal configurations of the AQx-2 dimers

Both motifs written by stage 1 are run here: 40 configurations each for
`H_backbone` and `J_end_group`.

Input is a dimer taken from the experimental crystal structure, so the packing
motif is already correct. The job here is only to sample thermal fluctuations
around it - the dynamic disorder that makes transfer integrals a distribution
rather than a single number.

Three things follow from starting at the experimental geometry:

* **No annealing.** An earlier version heated the pair to reorganise it out of
  a poor constructed placement; doing that here would destroy the very packing
  we went to the crystal to obtain.

* **Hydrogens are relaxed first, heavy atoms fixed.** X-ray hydrogen positions
  are placed geometrically and ride on their parent atoms (C-H 0.93-0.97 A in
  this CIF), far too short for a quantum calculation - the crystal geometry
  starts at fmax ~7.6 eV/A almost entirely because of it. Relaxing them with the
  heavy skeleton fixed repairs that without moving the packing.

* **The heavy skeleton is NOT relaxed.** In the crystal this pair is held in
  place by the surrounding lattice, which is absent here; relaxing to the
  gas-phase minimum would slide the molecules to an arrangement that is not the
  experimental one. The trajectory samples around the crystal geometry, and the
  drift column reports how far it wanders so you can see whether that
  assumption held.

## Input

`crystal_dimers/H_backbone_dimer.xyz` and `crystal_dimers/J_end_group_dimer.xyz`,
from stage 1.

## Output

Per motif, into `configs_<motif>/`:

| file | contents |
|---|---|
| `config_NNN_dimer.xyz` | both molecules, for the dimer SCF |
| `config_NNN_A.xyz` | monomer A alone, in the dimer's frame |
| `config_NNN_B.xyz` | monomer B alone, in the dimer's frame |

plus `manifest_<motif>.csv`, written incrementally. Everything is motif-tagged,
so the two runs cannot overwrite each other.

Each motif checkpoints itself: if a run is interrupted, rerun the cells and it
resumes from the last completed frame. Delete `checkpoint_<motif>.npz` to force
a fresh run of that motif.

Needs `torch`, `ase>=3.28` and `mace-torch`. A GPU is close to essential - 45 ps
of MACE dynamics per motif on a CPU is impractically slow.

In [ ]:
# ---- Settings -----------------------------------------------------------
import os

import numpy as np
import torch
from ase import Atoms, units
from ase.constraints import FixAtoms, FixCom
from ase.io import read, write
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import (Stationary, ZeroRotation,
                                         thermalize_momenta)
from ase.optimize import FIRE
from mace.calculators import mace_off

MOTIFS = ["H_backbone", "J_end_group"]

MODEL = "medium"
TEMPERATURE = 300.0         # K
TIMESTEP = 0.5              # fs; keeps the C-H stretches resolved
FRICTION = 0.01             # 1/fs; weak enough not to overdamp the stack
EQUIL_FS = 5000.0           # 5 ps, discarded
SAMPLE_SPACING_FS = 1000.0  # 1 ps between frames, past the librational
                            # correlation time, so frames are independent
N_CONFIGS = 40              # per motif

EQUIL_CHUNK_FS = 500.0      # checkpoint this often during equilibration

# Vacuum has no lattice holding the pair together. A flat-bottom restraint
# exerts no force at all until the centres of mass separate by more than their
# starting distance plus this margin, so it prevents the pair drifting apart
# without biasing the geometries actually sampled.
COM_MARGIN = 2.5            # A
K_RESTRAINT = 2.0           # eV/A^2

device = "cuda" if torch.cuda.is_available() else "cpu"

MANIFEST_HEADER = ("config,energy_eV,temperature_K,separation_A,slip_long_A,"
                   "slip_short_A,com_distance_A,closest_contact_A,"
                   "overlap_fraction,drift_A\n")

print(f"device {device}, motifs {', '.join(MOTIFS)}")
if device == "cpu":
    print("  WARNING: no GPU visible - this will be impractically slow")

In [ ]:
# ---- Helpers ------------------------------------------------------------
class FlatBottomCOM:
    """Harmonic restraint on the separation of two fragments' centres of mass,
    active only beyond r0. Below r0 it is exactly zero force."""

    def __init__(self, idx_a, idx_b, r0, k):
        self.a, self.b, self.r0, self.k = idx_a, idx_b, r0, k

    def adjust_positions(self, atoms, new):
        pass

    def adjust_forces(self, atoms, forces):
        m = atoms.get_masses()
        ma, mb = m[self.a], m[self.b]
        ca = (atoms.positions[self.a] * ma[:, None]).sum(0) / ma.sum()
        cb = (atoms.positions[self.b] * mb[:, None]).sum(0) / mb.sum()
        vec = cb - ca
        r = np.linalg.norm(vec)
        if r <= self.r0:
            return
        f = self.k * (r - self.r0) * vec / r
        forces[self.a] += f * (ma / ma.sum())[:, None]
        forces[self.b] -= f * (mb / mb.sum())[:, None]

    def get_removed_dof(self, atoms):
        return 0


def kabsch(P, Q):
    """Rotation+translation taking P onto Q (both N x 3)."""
    pc, qc = P.mean(axis=0), Q.mean(axis=0)
    U, _, Vt = np.linalg.svd((P - pc).T @ (Q - qc))
    d = np.sign(np.linalg.det(Vt.T @ U.T))
    R = Vt.T @ np.diag([1.0, 1.0, d]) @ U.T
    return R, qc - R @ pc


def descriptors(p, nA, heavy_A):
    """Separation along A's best-fit-plane normal, in-plane slip, centre of
    mass distance, closest contact, and the fraction of A in contact with B.

    Overlap is the descriptor that tracks how much cofacial contact there is,
    and that is what the transfer integral follows.
    """
    a, b = p[:nA][heavy_A], p[nA:][heavy_A]
    ca, cb = a.mean(axis=0), b.mean(axis=0)
    _, _, vt = np.linalg.svd(a - ca)
    delta = cb - ca
    d = np.linalg.norm(p[:nA, None, :] - p[None, nA:, :], axis=-1)
    return (abs(delta @ vt[2]), delta @ vt[0], delta @ vt[1],
            np.linalg.norm(delta), d.min(), (d.min(axis=1) < 5.0).mean())


def drift(p, p0, nA):
    """RMSD of monomer B from its starting placement, after superposing monomer
    A - i.e. how far the pair has moved relative to the crystal."""
    R, t = kabsch(p[:nA], p0[:nA])
    return np.sqrt((((p[nA:] @ R.T + t) - p0[nA:]) ** 2).sum(axis=1).mean())


def save_checkpoint(path, atoms, p0, equil_done, n_frames, motif):
    np.savez(path + ".tmp.npz", positions=atoms.get_positions(),
             momenta=atoms.get_momenta(), p0=p0,
             equil_done=equil_done, n_frames=n_frames, motif=motif)
    os.replace(path + ".tmp.npz", path)   # atomic, so an interruption
    # mid-write cannot leave a truncated checkpoint behind


# the MACE model is loaded once and reused for both motifs: float64 for the
# hydrogen relaxation, where the forces matter, and float32 for the dynamics,
# which is far faster and well inside the thermal noise being sampled
CALC = {}


def calculator(dtype):
    if dtype not in CALC:
        CALC[dtype] = mace_off(model=MODEL, default_dtype=dtype, device=device)
    return CALC[dtype]

## The per-motif run

One function, so the two motifs go through exactly the same steps. It is
resumable at frame granularity: a frame and its checkpoint are written together,
so an interrupted run picks up at the right place.

In [ ]:
# ---- One motif, start to finish -----------------------------------------
def run_motif(motif):
    input_xyz = f"crystal_dimers/{motif}_dimer.xyz"
    output_dir = f"configs_{motif}"
    manifest = f"manifest_{motif}.csv"
    checkpoint = f"checkpoint_{motif}.npz"
    md_log, relax_log = f"md_{motif}.log", f"relax_h_{motif}.log"
    os.makedirs(output_dir, exist_ok=True)

    print(f"\n{'=' * 70}\n{motif}\n{'=' * 70}", flush=True)

    # ---- Step 1: load the crystal dimer ---------------------------------
    atoms = read(input_xyz, format="xyz")
    n_atoms = len(atoms)
    nA = n_atoms // 2
    symbols = atoms.get_chemical_symbols()
    if symbols[:nA] != symbols[nA:]:
        raise SystemExit(f"{input_xyz}: the two halves are not the same "
                         f"molecule")
    heavy_A = np.array([i for i, s in enumerate(symbols[:nA]) if s != "H"])
    idx_A, idx_B = np.arange(nA), np.arange(nA, n_atoms)

    atoms.calc = calculator("float64")
    d0 = descriptors(atoms.get_positions(), nA, heavy_A)
    print(f"{n_atoms} atoms ({nA} per monomer); crystal geometry: contact "
          f"{d0[4]:.2f} A, COM {d0[3]:.2f} A, overlap {d0[5] * 100:.0f}%",
          flush=True)

    spacing = int(round(SAMPLE_SPACING_FS / TIMESTEP))
    equil_steps = int(round(EQUIL_FS / TIMESTEP))
    equil_chunk = int(round(EQUIL_CHUNK_FS / TIMESTEP))

    # ---- Step 2: resume, or relax the hydrogens -------------------------
    resumed = False
    if os.path.exists(checkpoint):
        # read inside a context manager and copy the arrays out: np.load
        # returns a lazy NpzFile that holds the file open, and on Windows that
        # blocks the atomic replace when the next checkpoint is written
        with np.load(checkpoint, allow_pickle=True) as ck:
            ck_motif, positions = str(ck["motif"]), ck["positions"].copy()
            momenta, p0 = ck["momenta"].copy(), ck["p0"].copy()
            equil_done, n_frames = int(ck["equil_done"]), int(ck["n_frames"])
        if ck_motif != motif:
            raise SystemExit(f"{checkpoint} is for motif {ck_motif}, not "
                             f"{motif}; delete it")
        atoms.set_positions(positions)
        atoms.set_momenta(momenta)
        resumed = True
        print(f"resuming: {equil_done}/{equil_steps} equilibration steps, "
              f"{n_frames}/{N_CONFIGS} frames saved", flush=True)
        if n_frames >= N_CONFIGS:
            print("already complete", flush=True)
            return manifest, d0
    else:
        atoms.set_constraint(FixAtoms(indices=[i for i, s in enumerate(symbols)
                                               if s != "H"]))
        print(f"start fmax {np.abs(atoms.get_forces()).max():.2f} eV/A "
              f"(X-ray riding hydrogens)", flush=True)
        FIRE(atoms, logfile=relax_log).run(fmax=0.05, steps=500)
        print(f"after H relaxation: fmax "
              f"{np.abs(atoms.get_forces()).max():.3f} eV/A", flush=True)
        atoms.set_constraint()
        p0 = atoms.get_positions().copy()
        equil_done, n_frames = 0, 0
        with open(manifest, "w") as f:
            f.write(MANIFEST_HEADER)

    r_flat = d0[3] + COM_MARGIN
    atoms.calc = calculator("float32")

    # ---- Step 3: thermostat ---------------------------------------------
    # FixCom holds the overall centre of mass: Langevin's own fixcm=True does
    # the same but is deprecated in ASE 3.28+ for not strictly sampling NVT.
    if not resumed:
        thermalize_momenta(atoms, temperature_K=TEMPERATURE)
        Stationary(atoms)
        ZeroRotation(atoms)
    atoms.set_constraint([FixCom(),
                          FlatBottomCOM(idx_A, idx_B, r_flat, K_RESTRAINT)])

    dyn = Langevin(atoms, timestep=TIMESTEP * units.fs,
                   temperature_K=TEMPERATURE, friction=FRICTION / units.fs,
                   fixcm=False, logfile=md_log, loginterval=200)

    # ---- Step 4: equilibrate --------------------------------------------
    if equil_done < equil_steps:
        print(f"equilibrating {EQUIL_FS / 1000:.0f} ps at {TEMPERATURE:.0f} K "
              f"(COM restraint beyond {r_flat:.1f} A)...", flush=True)
        while equil_done < equil_steps:
            dyn.run(min(equil_chunk, equil_steps - equil_done))
            equil_done += min(equil_chunk, equil_steps - equil_done)
            save_checkpoint(checkpoint, atoms, p0, equil_done, n_frames, motif)
        p = atoms.get_positions()
        d = descriptors(p, nA, heavy_A)
        print(f"  equilibrated: contact {d[4]:.2f} A, overlap "
              f"{d[5] * 100:.0f}%, drift {drift(p, p0, nA):.2f} A", flush=True)

    # ---- Step 5: production ---------------------------------------------
    # an explicit loop rather than dyn.attach, so a frame and its checkpoint
    # are written together and a resumed run picks up at exactly the right
    # frame
    print(f"production {spacing * N_CONFIGS * TIMESTEP / 1000:.0f} ps, one "
          f"frame every {SAMPLE_SPACING_FS / 1000:.1f} ps:", flush=True)
    while n_frames < N_CONFIGS:
        dyn.run(spacing)
        p = atoms.get_positions()
        e, t = atoms.get_potential_energy(), atoms.get_temperature()
        sep, sl, ss, com, contact, overlap = descriptors(p, nA, heavy_A)
        dr = drift(p, p0, nA)
        tag = f"config_{n_frames:03d}"
        elapsed = (equil_steps + (n_frames + 1) * spacing) * TIMESTEP / 1000
        note = (f"{tag}: {motif}, t={elapsed:.2f} ps, E={e:.4f} eV, "
                f"T={t:.1f} K, sep={sep:.3f} A, com={com:.3f} A, "
                f"contact={contact:.3f} A, overlap={overlap:.3f}, "
                f"drift={dr:.3f} A, n_A={nA}")
        # plain Atoms objects, and format pinned: ASE guesses format from the
        # filename and a name starting "config" matches DL_POLY CONFIG, not xyz
        write(f"{output_dir}/{tag}_dimer.xyz",
              Atoms(symbols=symbols, positions=p), format="xyz", comment=note)
        write(f"{output_dir}/{tag}_A.xyz",
              Atoms(symbols=symbols[:nA], positions=p[:nA]), format="xyz",
              comment=note + " [fragment A]")
        write(f"{output_dir}/{tag}_B.xyz",
              Atoms(symbols=symbols[nA:], positions=p[nA:]), format="xyz",
              comment=note + " [fragment B]")
        with open(manifest, "a") as f:      # appended per frame, so an
            f.write(f"{tag},{e:.6f},{t:.2f},{sep:.4f},{sl:.4f},{ss:.4f},"
                    f"{com:.4f},{contact:.4f},{overlap:.4f},{dr:.4f}\n")
        n_frames += 1
        save_checkpoint(checkpoint, atoms, p0, equil_steps, n_frames, motif)
        print(f"  {tag}  E {e:11.3f}  T {t:5.1f}  contact {contact:4.2f}  "
              f"overlap {overlap * 100:3.0f}%  drift {dr:4.2f}", flush=True)

    return manifest, d0

## Run both motifs

Roughly 45 ps of dynamics per motif. Watch the `drift` column: it is the RMSD of
monomer B from its crystal placement after superposing monomer A, so it says
whether the pair is still sampling the experimental packing or has slid to
something else in vacuum. `J_end_group` has about half the contact area of
`H_backbone`, so expect it to wander more.

In [ ]:
# ---- Run ----------------------------------------------------------------
results = {}
for motif in MOTIFS:
    results[motif] = run_motif(motif)

print("\ndone")

In [ ]:
# ---- Summary and sanity checks ------------------------------------------
for motif, (manifest, d0) in results.items():
    rows = np.atleast_1d(np.genfromtxt(manifest, delimiter=",", names=True))
    print(f"\n{motif}: {len(rows)} configurations in configs_{motif}/")
    print(f"  contact   {rows['closest_contact_A'].mean():.2f} +/- "
          f"{rows['closest_contact_A'].std():.2f} A (crystal {d0[4]:.2f})")
    print(f"  overlap   {rows['overlap_fraction'].mean() * 100:.0f} +/- "
          f"{rows['overlap_fraction'].std() * 100:.0f}% "
          f"(crystal {d0[5] * 100:.0f}%)")
    print(f"  drift     {rows['drift_A'].mean():.2f} +/- "
          f"{rows['drift_A'].std():.2f} A (max {rows['drift_A'].max():.2f})")

    if rows["drift_A"].max() > 2.0:
        print("  WARNING: the pair has moved well away from the crystal "
              "packing; these frames no longer sample the experimental motif")
    if rows["overlap_fraction"].mean() < 0.7 * d0[5]:
        print("  WARNING: overlap has dropped substantially below the crystal "
              "value - the molecules are sliding apart in vacuum")
    if rows["com_distance_A"].max() > d0[3] + COM_MARGIN:
        print("  WARNING: COM distance passed the restraint - the pair is "
              "being held together rather than staying together")